# Chapter 7: Instruction Finetuning

This notebook covers finetuning models to follow instructions:
- Instruction-following datasets
- Prompt engineering
- Training on (instruction, response) pairs
- Evaluation and inference

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import tiktoken
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 7.1: Instruction-Following Dataset

class InstructionDataset(Dataset):
    """Dataset for instruction finetuning"""
    
    def __init__(self, data, tokenizer, max_length=512):
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length
    
    def __len__(self):
        return len(self.data)
    
    def __getitem__(self, idx):
        item = self.data[idx]
        
        # Format: instruction + input -> response
        instruction = item['instruction']
        input_text = item.get('input', '')
        response = item['response']
        
        # Create prompt
        if input_text:
            prompt = f"Instruction: {instruction}\n\nInput: {input_text}\n\nResponse: "
        else:
            prompt = f"Instruction: {instruction}\n\nResponse: "
        
        # Full text
        full_text = prompt + response
        
        # Tokenize
        prompt_tokens = self.tokenizer.encode(prompt)
        full_tokens = self.tokenizer.encode(full_text)
        
        # Truncate
        if len(full_tokens) > self.max_length:
            full_tokens = full_tokens[:self.max_length]
        
        # Pad
        padding_length = self.max_length - len(full_tokens)
        full_tokens = full_tokens + [50256] * padding_length
        
        # Create attention mask (1 for real tokens, 0 for padding)
        attention_mask = [1] * (self.max_length - padding_length) + [0] * padding_length
        
        # Create labels (only compute loss on response tokens)
        labels = full_tokens.copy()
        # Mask out prompt tokens
        for i in range(min(len(prompt_tokens), len(labels))):
            labels[i] = -100  # Ignore index
        
        return {
            'input_ids': torch.tensor(full_tokens, dtype=torch.long),
            'labels': torch.tensor(labels, dtype=torch.long),
            'attention_mask': torch.tensor(attention_mask, dtype=torch.float)
        }

tokenizer = tiktoken.get_encoding("gpt2")
dataset = InstructionDataset(instruction_data, tokenizer)

# Test a sample
sample = dataset[0]
print(f"Sample input shape: {sample['input_ids'].shape}")
print(f"Sample has {(sample['labels'] != -100).sum()} response tokens")

## 7.2: Instruction Finetuning with Masked Loss

def instruction_finetune_epoch(model, train_loader, optimizer, device):
    """Finetune for one epoch"""
    model.train()
    total_loss = 0
    
    pbar = tqdm(train_loader, desc="Finetuning")
    for batch in pbar:
        input_ids = batch['input_ids'].to(device)
        labels = batch['labels'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        
        # Forward pass
        # Need to modify model to return hidden states for classification
        # For now, assume we have logits output
        
        # This is pseudocode - you would use actual model forward
        # logits = model(input_ids)
        
        # Compute masked loss
        # loss = instruction_finetuning_loss(logits, labels, attention_mask)
        
        # Backward pass
        # optimizer.zero_grad()
        # loss.backward()
        # torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        # optimizer.step()
        
        # total_loss += loss.item()
        
        # pbar.set_postfix({'loss': f'{loss.item():.4f}'})
        pass

print("Instruction finetuning epoch function defined")

# Different prompt templates
PROMPT_TEMPLATES = {
    'basic': "Instruction: {instruction}\nResponse: ",
    
    'detailed': """Below is an instruction that describes a task. Write a response that appropriately completes the request.
    
### Instruction:
{instruction}

### Response:
""",
    
    'with_context': """Instruction: {instruction}

Context: {input}

Response: """,
    
    'few_shot': """You are a helpful assistant. Answer the following question.

Q: What is 2+2?
A: 4

Q: {instruction}
A: """
}

print("Prompt templates defined")
for name, template in PROMPT_TEMPLATES.items():
    print(f"\n{name}:")
    print(template[:80] + "...")

def generate_instruction_response(model, tokenizer, instruction, device, max_length=128, temperature=0.7):
    """
    Generate a response for a given instruction.
    """
    # Create prompt
    prompt = f"Instruction: {instruction}\n\nResponse: "
    
    # Tokenize
    input_ids = torch.tensor(tokenizer.encode(prompt)).unsqueeze(0).to(device)
    
    # Generate
    generated_ids = input_ids.clone()
    
    with torch.no_grad():
        for _ in range(max_length):
            logits = model(generated_ids)
            # Take last token logits
            last_logits = logits[:, -1, :] / temperature
            
            # Sample next token
            probs = torch.softmax(last_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            
            # Append
            generated_ids = torch.cat([generated_ids, next_token], dim=-1)
            
            # Stop if EOS
            if next_token.item() == tokenizer.eot_token:
                break
    
    # Decode
    response = tokenizer.decode(generated_ids[0].cpu().numpy())
    return response

print("Generation function defined")

# Evaluation metrics for instruction following
print("""
Evaluation Metrics for Instruction Following:

1. **BLEU Score**: Compares generated text with reference
   - Measures n-gram overlap
   - Common in machine translation

2. **ROUGE Score**: Recall-Oriented Understudy for Gisting Evaluation
   - ROUGE-1: Unigram overlap
   - ROUGE-2: Bigram overlap
   - ROUGE-L: Longest common subsequence

3. **METEOR**: Metric for Evaluation of Translation with Explicit ORdering
   - Considers synonyms and word order

4. **Human Evaluation**:
   - Relevance: Does response answer the instruction?
   - Coherence: Is the response well-structured?
   - Factuality: Is the information correct?
   - Completeness: Does it fully address the instruction?
""")

## 7.6: Best Practices for Instruction Finetuning

## 7.7: Common Challenges and Solutions

## 7.8: Advanced Techniques

## Summary

Instruction finetuning transforms pretrained models into helpful assistants:
1. Create diverse instruction-response datasets
2. Use consistent prompt templates
3. Train with masked loss (only on responses)
4. Evaluate with appropriate metrics
5. Use advanced techniques (RLHF, DPO, etc.) for production systems

This is the final step in building a complete LLM!